# p3_f5 — huan luyen + eval da nhanh trong MOT lan chay

**Run All roi de day (~3h).** Khong phai doi JOB, khong phai upload checkpoint, khong phai
mo notebook thu hai. Can: GPU + Add Input `forget-mi-data` va `forget-mi-models-full`.

| Cell | Viec | ~gio |
|---|---|---|
| 1 | setup + tu kiem eval_multimodal | 0,05 |
| 2 | do path + in cau hinh F5 | 0,02 |
| 3 | **huan luyen** p3_f5, 30 epoch | 2,7 |
| 4 | **eval da nhanh** (img/txt/fuse) tren checkpoint vua tao, tu thu lai FP32 neu NaN | 0,25 |
| 5 | bang ket qua + TU KIEM | 0,01 |

## F5 la gi

`f5 = f4` nhung **bo LoRA tren `img_model.fc1`**. Ly do: `fc1` khong phai lop bieu dien ma
la **dau phan loai** cua nhanh anh —

    joint_img_txt/model.py:113   self.fc1 = nn.Linear(768, output_channels)
    joint_img_txt/model.py:182   logits = self.fc1(z)      -> outputs[1] = img_logits

tuc no sinh ra chinh cai logit ma Df-AUC / MIA / forget_ce / test_ce deu doc. Gan LoRA len
no cho phep ha IHL bang cach xoay rieng ma tran tuyen tinh cuoi **ma khong doi bieu dien z**
— va trai voi rang buoc LoKU Sec.3.5 (classifier head dong bang) ma
`config_advanced_kaggle.yaml` dang tuyen bo. Assert "LoRA-only" khong bat duoc, vi adapter
dat tren head van dung la tham so LoRA.

F5 khac F4 **dung mot khoa** nen tach bach duoc:

- **F5 ~ F4** -> loi ich den tu suc chua BIEU DIEN (them layer5). Chot F5, hop le hoan toan.
- **F5 << F4** -> phan lon loi the cua F4 den tu chinh DAU PHAN LOAI. F4 khong dung lam ket
  qua chinh duoc, chi vao muc Han che.

Giu `loku_subtract_scale=1.5` (du da biet no gan nhu vo dung: F2 ~ F3) de phep so sanh voi
F4 chi lech dung mot khoa. Doi lai, F5 co the dinh dung loi NaN nhanh van ban nhu F3 — Cell 4
tu xu ly bang cach chay lai o FP32.

## Cau hinh (moi lambda)

| | F3 | F4 | **F5** |
|---|---:|---:|---:|
| w_UR / w_UU / w_MU / w_MR | 1/3, 1/3, 1/6, 1/6 | (cung) | (cung) |
| lambda_KD | 0 | 0 | 0 |
| lambda_CE | 0.25 | 0.25 | **0.25** |
| lambda_IHL | 5.0 | 5.0 | **5.0** |
| loku_subtract_scale | 1.5 | 1.5 | **1.5** |
| loku_image_subtract_scale | 1.0 | 1.0 | **1.0** |
| lora_image_last_k_blocks | 2 | 3 | **3** |
| lora_image_include_fc1 | 0 | 1 | **0** |


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
for f in ['training/forgetmi_p3_cand.py','training/eval_multimodal.py','training/adv_common.py']:
    assert os.path.exists(f), f'Thieu {f} -> git push code moi truoc'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))

# Tu kiem phan thuan-mang cua eval_multimodal (30 giay, truoc khi ton 2,7h GPU)
print('\n--- tu kiem eval_multimodal ---')
subprocess.run(['python','tools/test_eval_multimodal.py'],check=True)


In [ ]:
# Cell 2: path + cau hinh F5
import glob, os
SEED   = 42
EPOCHS = 30
RID    = f'p3_f5_s{SEED}'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

CONFIG='config_advanced_kaggle.yaml'
DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
assert DATA and MOD,'Add Input: forget-mi-data + forget-mi-models-full'
BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
gh=[b for b in bins(MOD) if 'model_retrained_3per' in b]
assert gh,'Khong thay model_retrained_3per'
GOLD=os.path.dirname(gh[0])
TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
FORGET='./data_splits/forget_set_3per.csv'
for n,p in {'BASE':BASE,'GOLD':GOLD,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

OUT   = f'/kaggle/working/kltn_p3_f5_s{SEED}'
OD    = f'{OUT}/{RID}'
CKPT  = f'{OD}/checkpoints/latest.pt'          # E30, do adv_common.save_ckpt ghi moi epoch
R_TRAIN = '/kaggle/working/results_p3_f5.csv'
R_MM    = '/kaggle/working/results_multimodal_f5.csv'
HIST    = f'/kaggle/working/perepoch_{RID}.csv'

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'use_noise':1}
# P3-NoKD-More: cau hinh da khoa tu MIMIC 3%
MLP_TXT='attention.output.dense|intermediate.dense|output.dense'
MORE={'lora_extra_target_modules':MLP_TXT}

# --- F5: giong f4 tru dung mot khoa lora_image_include_fc1 ---
# fc1 = dau phan loai anh (model.py:113/182 -> outputs[1] = img_logits). Gan LoRA len no
# la cho phuong phap sua thang thuoc do + trai rang buoc LoKU Sec.3.5 (head dong bang).
F5={'lambda_ihl':5.0, 'lambda_ce':0.25,
    'loku_subtract_scale':1.5, 'loku_image_subtract_scale':1.0,
    'lora_image_last_k_blocks':3, 'lora_image_include_fc1':0}
# Cac khoa PHAI khai lai luc dung lai W* tu checkpoint chi-chua-LoRA. Hai he so tru FILA
# khong suy duoc tu ten khoa (khac lora_extra/image_last_k) ma lai quyet dinh
# W* = W - gamma*B*A* -> sai la moi so sai AM THAM, khong assert nao bat.
F5_REBUILD={k:F5[k] for k in ['loku_subtract_scale','loku_image_subtract_scale',
                              'lora_image_last_k_blocks','lora_image_include_fc1']}

print('run id   :',RID)
print('BASE     :',BASE)
print('GOLD     :',GOLD)
print('ckpt se o:',CKPT)
print('\n--- F5 overrides ---')
for k,v in F5.items(): print(f'   {k:32} = {v}')
print('\nDoi chieu khi chay: Trainable phai LON HON 1,451,008 (f3) va NHO HON 1,495,072 (f4).')


In [ ]:
# Cell 3: HUAN LUYEN p3_f5 (30 epoch, ~2,7h)
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

ovr=dict(COMMON); ovr.update(MORE)
ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,'results_csv_path':R_TRAIN,
            'ce_selector':1,'s4_delta':0.15,'history_csv_path':HIST})
ovr.update(F5)
cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
     '--scheme','uni_nokd','--ablate','none','--fresh','--override',
     ','.join(f'{k}={v}' for k,v in ovr.items())]

print('='*72+f'\nTRAIN {RID}\n'+'='*72)
t0=time.time(); TRAIN_OK=False
try:
    subprocess.run(cmd,env=env,check=True)
    TRAIN_OK=True
    print(f'OK train  {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL train rc=',e.returncode)
print('checkpoint ton tai:',os.path.exists(CKPT))


In [ ]:
# Cell 4: EVAL DA NHANH tren checkpoint vua tao (img / txt / fuse)
# Chay FP16 truoc (cung do chinh xac voi og/re/f4 da do) -> neu nhanh van ban tran FP16
# va cho NaN nhu f3 thi TU DONG chay lai o FP32, ghi them nhan f5_fp32.
import os, subprocess, time
import pandas as pd

def run_eval(label, extra):
    ovr=dict(COMMON); ovr.update(MORE); ovr.update(F5_REBUILD)
    ovr['output_dir']=f'{OUT}/_mm'; ovr.update(extra)
    cmd=['python','training/eval_multimodal.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','p3_lora','--model_path',CKPT,
         '--out_csv',R_MM,'--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('='*72+f'\nEVAL {label}\n'+'='*72)
    t0=time.time()
    try:
        subprocess.run(cmd,env=env,check=True)
        print(f'OK {label}  {(time.time()-t0)/60:.1f} phut'); return True
    except subprocess.CalledProcessError as e:
        print(f'FAIL {label} rc={e.returncode}'); return False

if not os.path.exists(CKPT):
    print('Khong co checkpoint -> bo qua eval. Xem lai Cell 3.')
else:
    run_eval('f5', {})
    need_fp32=False
    if os.path.exists(R_MM):
        d=pd.read_csv(R_MM)
        d=d[d['label']=='f5']
        need_fp32=bool(d[d['view'].isin(['txt','fuse'])]['Df_AUC'].isna().any())
    if need_fp32:
        print('\ntxt/fuse ra NaN o FP16 (giong f3) -> chay lai FP32...')
        run_eval('f5_fp32', {'eval_autocast':0})
    else:
        print('\nFP16 du dung, khong can chay lai FP32.')


In [ ]:
# Cell 5: BANG KET QUA + TU KIEM
import os
import pandas as pd
pd.set_option('display.width',250)

print('===== HUAN LUYEN (E30 = checkpoint_kind "last") =====')
train_row=None
if os.path.exists(R_TRAIN):
    dt=pd.read_csv(R_TRAIN)
    cols=[c for c in ['id','checkpoint_kind','selected_epoch','Forget_AUC','Forget_Macro_F1',
                      'Test_AUC','Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce',
                      '1_minus_Sim','trainable_params','trainable_ratio','core_seconds']
          if c in dt.columns]
    print(dt[cols].to_string(index=False))
    last=dt[dt['checkpoint_kind']=='last']
    if len(last): train_row=last.iloc[-1]
else:
    print('chua co',R_TRAIN)

print('\n===== EVAL DA NHANH =====')
if os.path.exists(R_MM):
    dm=pd.read_csv(R_MM)
    c=['label','view','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper',
       'member_ce','nonmember_ce','forget_ce']
    print(dm[[x for x in c if x in dm.columns]].to_string(index=False))
else:
    print('chua co',R_MM)

print('\n===== TU KIEM: dong view=img phai TRUNG ket qua huan luyen =====')
if train_row is not None and os.path.exists(R_MM):
    img=pd.read_csv(R_MM)
    img=img[(img['label']=='f5') & (img['view']=='img')]
    if len(img):
        r=img.iloc[-1]
        for name,a,b in [('Df-AUC', r['Df_AUC'], train_row['Forget_AUC']),
                         ('Dt-AUC', r['Dt_AUC'], train_row['Test_AUC']),
                         ('MIA',    r['MIA'],    train_row['MIA']),
                         ('forget-CE', r['forget_ce'], train_row['forget_ce'])]:
            ok = abs(float(a)-float(b)) < 0.005
            print(f'  {name:10} eval {float(a):7.4f}  vs  train {float(b):7.4f}   '
                  + ('TRUNG' if ok else '*** LECH -> dung tin so txt/fuse ***'))
    else:
        print('  khong co dong view=img cho f5')
else:
    print('  thieu du lieu de doi chieu')

print('\n===== SO VOI F3 / F4 (nhanh anh, E30) =====')
print('  mo hinh   Df-AUC   Dt-AUC     MIA  forget-CE  test-CE      tham so')
print('  gold       0.498    0.615   0.423      4.736    3.046  113,238,164')
print('  F3         0.590    0.692   0.418      3.477    2.256    1,451,008')
print('  F4         0.528    0.667   0.368      4.506    2.369    1,495,072')
print('  F5 ~ F4  -> loi ich den tu suc chua bieu dien (layer5): chot F5, hop le.')
print('  F5 << F4 -> phan lon loi the cua F4 den tu viec chinh dau phan loai fc1.')

print('\nTAI VE: results_p3_f5.csv - results_multimodal_f5.csv - perepoch_p3_f5_s42.csv')
print('        + kltn_p3_f5_s42/**/checkpoints/latest.pt (giu lai de do lai sau nay)')
